# MARL flocking
---

In [ ]:
!pip install pettingzoo
!pip install gymnasium
!pip install torch
!pip install numpy
!pip install onnxscript

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from pettingzoo import ParallelEnv
from gymnasium.spaces import Box

## Enviroment

In [ ]:
class StarlingFlockEnv(ParallelEnv):

    def __init__(
        self,
        num_birds=32,
        world_size=20.0,
        max_speed=1.0,
        neighbor_radius=3.0,
        max_steps=500,
    ):

        self.num_birds = num_birds
        self.world_size = world_size
        self.max_speed = max_speed
        self.neighbor_radius = neighbor_radius
        self.max_steps = max_steps

        self.agents = [f"bird_{i}" for i in range(num_birds)]

        # observation:
        # [self_vel_x,
        #  self_vel_y,
        #  avg_neighbor_dx,
        #  avg_neighbor_dy,
        #  avg_neighbor_vel_x,
        #  avg_neighbor_vel_y]
        obs_dim = 6

        # action:
        # delta velocity x, y
        act_dim = 2

        self.observation_spaces = {
            a: Box(
                low=-np.inf,
                high=np.inf,
                shape=(obs_dim,),
                dtype=np.float32
            )
            for a in self.agents
        }

        self.action_spaces = {
            a: Box(
                low=-1.0,
                high=1.0,
                shape=(act_dim,),
                dtype=np.float32
            )
            for a in self.agents
        }

    def reset(self, seed=None, options=None): # reset env to start new epoch

        self.step_count = 0

        self.positions = {
            a: np.random.uniform(0, self.world_size, size=(2,)).astype(np.float32)
            for a in self.agents
        }

        self.velocities = {
            a: np.random.uniform( -0.5, 0.5, size=(2,)).astype(np.float32)
            for a in self.agents
        }

        observations = {
            a: self._get_obs(a)
            for a in self.agents
        }

        infos = {
            a: {}
            for a in self.agents
        }

        return observations, infos


    def step(self, actions):

        self.step_count += 1

        rewards = {}
        observations = {}
        terminations = {}
        truncations = {}
        infos = {}

        for agent, action in actions.items():

            action = np.clip(action, -1.0, 1.0)

            self.velocities[agent] += action * 0.05

            speed = np.linalg.norm(self.velocities[agent])

            if speed > self.max_speed:
                self.velocities[agent] /= speed
                self.velocities[agent] *= self.max_speed

        for agent in self.agents:

            self.positions[agent] += self.velocities[agent]

            # wrap around world
            self.positions[agent] %= self.world_size

        for agent in self.agents:

            observations[agent] = self._get_obs(agent)

            rewards[agent] = self._compute_reward(agent)

            terminations[agent] = False

            truncations[agent] = (
                self.step_count >= self.max_steps
            )

            infos[agent] = {}

        return (
            observations,
            rewards,
            terminations,
            truncations,
            infos
        )


    def _neighbors(self, agent):

        p = self.positions[agent]

        neighbors = []

        for other in self.agents:

            if other == agent:
                continue

            d = self.positions[other] - p

            dist = np.linalg.norm(d)

            if dist < self.neighbor_radius:
                neighbors.append(other)

        return neighbors

    def _get_obs(self, agent):

        neighbors = self._neighbors(agent)

        self_vel = self.velocities[agent]

        if len(neighbors) == 0:

            return np.concatenate([
                self_vel,
                np.zeros(4, dtype=np.float32)
            ]).astype(np.float32)

        rel_positions = []
        rel_vels = []

        for n in neighbors:

            rel_positions.append(
                self.positions[n] - self.positions[agent]
            )

            rel_vels.append(
                self.velocities[n]
            )

        avg_rel_pos = np.mean(rel_positions, axis=0)
        avg_rel_vel = np.mean(rel_vels, axis=0)

        obs = np.concatenate([
            self_vel,
            avg_rel_pos,
            avg_rel_vel
        ])

        return obs.astype(np.float32)


    def _compute_reward(self, agent):

        neighbors = self._neighbors(agent)

        if len(neighbors) == 0:
            return -1.0

        # cohesion
        center = np.mean(
            [self.positions[n] for n in neighbors],
            axis=0
        )

        dist_to_center = np.linalg.norm(
            self.positions[agent] - center
        )

        cohesion_reward = 1.0 / (1.0 + dist_to_center)

        return cohesion_reward

## Model

In [ ]:
class SharedPolicy(nn.Module):

    def __init__(self, obs_dim=6, action_dim=2):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(obs_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
            nn.Tanh()
        )

    def forward(self, x):
        return self.net(x)

## Learning process

In [ ]:
device = torch.device("cpu")

env = StarlingFlockEnv(
    num_birds=32
)

model = SharedPolicy().to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

gamma = 0.99

episodes = 100

for episode in range(episodes):

    obs, infos = env.reset()

    episode_reward = 0.0

    log_probs = []
    rewards_buffer = []

    done = False

    while not done:

        actions = {}

        step_log_probs = []

        for agent in env.agents:

            obs_tensor = torch.tensor(
                obs[agent],
                dtype=torch.float32
            ).unsqueeze(0)

            mean_action = model(obs_tensor)

            dist = torch.distributions.Normal(
                mean_action,
                0.1
            )

            action = dist.sample()

            log_prob = dist.log_prob(action).sum()

            actions[agent] = (
                action.squeeze(0)
                .detach()
                .numpy()
            )

            step_log_probs.append(log_prob)

        next_obs, rewards, terms, truncs, infos = env.step(actions)

        avg_reward = np.mean(
            list(rewards.values())
        )

        rewards_buffer.append(avg_reward)

        log_probs.append(
            torch.stack(step_log_probs).mean()
        )

        episode_reward += avg_reward

        done = all(truncs.values())

        obs = next_obs

    returns = []

    G = 0

    for r in reversed(rewards_buffer):

        G = r + gamma * G

        returns.insert(0, G)

    returns = torch.tensor(
        returns,
        dtype=torch.float32
    )

    returns = (returns - returns.mean()) / returns.std()

    loss = 0

    for log_prob, G in zip(log_probs, returns):

        loss += -log_prob * G

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    print(
        f"Episode {episode} | "
        f"Reward: {episode_reward:.2f}"
    )

In [ ]:
dummy_input = torch.randn(1, 6)

torch.onnx.export(
    model,
    dummy_input,
    "starling_policy.onnx",
    opset_version=11,
    input_names=["obs"],
    output_names=["action"],
)

print("\nExported: starling_policy.onnx")